# freqgen — Spectral Matching Attack vs SPAI (Full Colab Experiment)

**End-to-end experiment on the SPAI (CVPR 2025) exact evaluation dataset.**

What this notebook does:
1. Installs SPAI + downloads its weights
2. Downloads **Synthbuster** (fakes: SD 1.4, SDXL) + **RAISE-1k** (reals)
3. Builds the real spectral target from RAISE-1k
4. Runs the **spectral matching attack** on Synthbuster fakes
5. Runs **SPAI inference** on raw fakes and matched fakes
6. Prints the evasion table: does the attack fool SPAI?

Requires a **GPU runtime** (T4 free tier is fine). Go to:
`Runtime → Change runtime type → T4 GPU`

## 1. Install SPAI

In [ ]:
import os
# Clone SPAI repo
if not os.path.exists('/content/spai'):
    !git clone https://github.com/mever-team/spai.git /content/spai
%cd /content/spai
!pip install -r requirements.txt -q
print('SPAI installed')

In [ ]:
# Download SPAI weights via gdown (Google Drive)
!pip install gdown -q
import os
os.makedirs('/content/spai/weights', exist_ok=True)
if not os.path.exists('/content/spai/weights/spai.pth'):
    !gdown 1vvXmZqs6TVJdj8iF1oJ4L_fcgdQrp_YI -O /content/spai/weights/spai.pth
print('Weights ready:', os.path.getsize('/content/spai/weights/spai.pth')//1_000_000, 'MB')

## 2. Download data

**Synthbuster** (~12 GB total, we grab only SD1.4 + SDXL subset) and **RAISE-1k** (reals).

We use the SPAI repo's own CSV files so paths match exactly.

In [ ]:
import os, zipfile, urllib.request

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

# ---- RAISE-1k (reals) ----
RAISE_DIR = f'{DATA_DIR}/RAISE-1k/tiff'
if not os.path.exists(RAISE_DIR) or len(os.listdir(RAISE_DIR)) < 100:
    print('Downloading RAISE-1k...')
    !wget -q 'https://loki.disi.unitn.it/RAISE/download/RAISE_1k_tiff.zip' -O /tmp/raise.zip
    !unzip -q /tmp/raise.zip -d {DATA_DIR}/RAISE-1k
    print('RAISE-1k done')
else:
    print('RAISE-1k already present:', len(os.listdir(RAISE_DIR)), 'files')

In [ ]:
# ---- Synthbuster (only SD1.4 + SDXL subfolders) ----
# Full zip is 12GB; we stream-extract only needed subfolders.
import zipfile, io, urllib.request

SYNTH_ROOT = f'{DATA_DIR}/synthbuster'
NEEDED = ['stable-diffusion-1-4', 'stable-diffusion-xl']

already = all(
    os.path.exists(f'{SYNTH_ROOT}/{g}') and len(os.listdir(f'{SYNTH_ROOT}/{g}')) >= 100
    for g in NEEDED
)

if not already:
    print('Downloading Synthbuster (full zip ~12 GB) — this takes ~10–15 min on Colab...')
    ZENODO_URL = 'https://zenodo.org/records/10066460/files/synthbuster.zip/content'
    os.makedirs(SYNTH_ROOT, exist_ok=True)
    # Stream-download and extract only needed folders
    TMP = '/tmp/synthbuster.zip'
    !wget -q '{ZENODO_URL}' -O {TMP}
    print('Extracting SD1.4 and SDXL subfolders only...')
    with zipfile.ZipFile(TMP) as z:
        members = [m for m in z.namelist()
                   if any(g in m for g in NEEDED)]
        z.extractall(DATA_DIR, members=members)
    os.remove(TMP)
    print('Synthbuster extracted')
else:
    print('Synthbuster already present')

for g in NEEDED:
    p = f'{SYNTH_ROOT}/{g}'
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f'  {g}: {n} images')

## 3. Spectral helpers (from freqgen/src/spectral.py)

In [ ]:
import numpy as np
from PIL import Image

LOW_MID_EDGE, MID_HIGH_EDGE = 20, 60
SIZE = 256

def load_gray(path):
    return np.asarray(Image.open(path).convert('L').resize((SIZE,SIZE)),dtype=np.float64)

def _radius_map(h, w):
    cy, cx = h//2, w//2
    y, x = np.ogrid[:h, :w]
    return np.round(np.sqrt((y-cy)**2+(x-cx)**2)).astype(int)

def radial_profile(ch):
    F = np.fft.fftshift(np.fft.fft2(ch)); mag = np.abs(F)
    r = _radius_map(*ch.shape); mr = min(ch.shape)//2
    t = np.bincount(r.ravel(), weights=mag.ravel())
    c = np.bincount(r.ravel())
    return t[:mr]/np.maximum(c[:mr],1)

def _match_channel(ch, target, gain_clip=(0.1,12.0), smooth=3, preserve_dc=True):
    F = np.fft.fftshift(np.fft.fft2(ch)); r = _radius_map(*ch.shape); mr = len(target)
    gain = target/(radial_profile(ch)+1e-12)
    if smooth>1: gain = np.convolve(gain,np.ones(smooth)/smooth,mode='same')
    gain = np.clip(gain,*gain_clip)
    if preserve_dc: gain[0] = 1.0
    g = gain[np.clip(r,0,mr-1)]; g[r>=mr] = 1.0
    if preserve_dc: g[r==0] = 1.0
    return np.fft.ifft2(np.fft.ifftshift(F*g)).real

def spectral_match(img, target):
    return np.clip(_match_channel(img,target),0,255)

def spectral_report(ch):
    prof = radial_profile(ch)
    freqs = np.arange(1,len(prof))
    slope,_ = np.polyfit(np.log(freqs+1e-8),np.log(prof[1:]+1e-8),1)
    return {'low':float(prof[:LOW_MID_EDGE].mean()),
            'mid':float(prof[LOW_MID_EDGE:MID_HIGH_EDGE].mean()),
            'high':float(prof[MID_HIGH_EDGE:].mean()),
            'slope':float(slope),'profile':prof}

print('Spectral helpers ready')

## 4. Build real spectral target + verify the ~21x gap

Compute the average radial profile of RAISE-1k reals using SPAI's CSV.

In [ ]:
import pandas as pd
from pathlib import Path

# Load SPAI's exact CSV for RAISE-1k reals
!wget -q https://raw.githubusercontent.com/mever-team/spai/main/data/real_raise.csv \
     -O /content/real_raise.csv
!wget -q https://raw.githubusercontent.com/mever-team/spai/main/data/fake_sd14.csv \
     -O /content/fake_sd14.csv
!wget -q https://raw.githubusercontent.com/mever-team/spai/main/data/fake_sdxl.csv \
     -O /content/fake_sdxl.csv

real_df  = pd.read_csv('/content/real_raise.csv')
fake_df  = pd.read_csv('/content/fake_sd14.csv')
fakex_df = pd.read_csv('/content/fake_sdxl.csv')
print(f'real_raise: {len(real_df)} | fake_sd14: {len(fake_df)} | fake_sdxl: {len(fakex_df)}')

In [ ]:
import os
# Resolve full image paths (CSV paths are relative to DATA_DIR)
def resolve(df, root=DATA_DIR, n=100):
    paths = []
    for p in df['image']:
        fp = os.path.join(root, p)
        if os.path.exists(fp):
            paths.append(fp)
        if len(paths) >= n:
            break
    return paths

real_paths  = resolve(real_df,  n=100)
fake_paths  = resolve(fake_df,  n=100)
fakex_paths = resolve(fakex_df, n=100)
print(f'Resolved: {len(real_paths)} real | {len(fake_paths)} SD1.4 | {len(fakex_paths)} SDXL')

In [ ]:
from tqdm.notebook import tqdm

# Build real target (average radial profile over 100 RAISE-1k images)
print('Computing real target profile...')
real_profiles = [radial_profile(load_gray(p)) for p in tqdm(real_paths)]
target = np.mean(real_profiles, axis=0)

# Verify the gap on real Synthbuster SD1.4 fakes
print('\nComputing spectral reports...')
rep_real = spectral_report(load_gray(real_paths[0]))
rep_fake = spectral_report(load_gray(fake_paths[0]))

real_high_mean = np.mean([spectral_report(load_gray(p))['high'] for p in tqdm(real_paths[:30])])
fake_high_mean = np.mean([spectral_report(load_gray(p))['high'] for p in tqdm(fake_paths[:30])])

print(f'\nHigh-band mean  real: {real_high_mean:.1f}')
print(f'High-band mean  fake: {fake_high_mean:.1f}')
print(f'Gap (real/fake): {real_high_mean/fake_high_mean:.1f}x  <- the freqgen finding on real data')

## 5. Run the spectral matching attack

Rewrite each fake's Fourier magnitude to match the real target. Phase untouched.

In [ ]:
import os
MATCHED_DIR = '/content/data/synthbuster_matched/stable-diffusion-1-4'
os.makedirs(MATCHED_DIR, exist_ok=True)

print('Applying spectral matching to SD1.4 fakes...')
matched_paths = []
for p in tqdm(fake_paths):
    img = load_gray(p)
    m   = spectral_match(img, target)
    out = os.path.join(MATCHED_DIR, os.path.basename(p))
    Image.fromarray(m.astype(np.uint8)).save(out)
    matched_paths.append(out)

print(f'Saved {len(matched_paths)} matched fakes to {MATCHED_DIR}')

# Verify gap closed
matched_high_mean = np.mean([spectral_report(load_gray(p))['high'] for p in tqdm(matched_paths[:30])])
print(f'\nHigh-band gap real/fake:    {real_high_mean/fake_high_mean:.1f}x')
print(f'High-band gap real/matched: {real_high_mean/matched_high_mean:.2f}x  <- attack worked')

## 6. Hand-crafted detector evasion (CPU baseline)

In [ ]:
# Logistic regression on radial features — should be 100% evaded
def radial_features(ch): return np.log(radial_profile(ch)+1e-8)

class LogReg:
    def __init__(s,lr=0.5,ep=2000,l2=1e-3): s.lr,s.ep,s.l2=lr,ep,l2
    def fit(s,X,y):
        y=np.asarray(y,float); s.mu=X.mean(0); s.sd=X.std(0)+1e-8
        Xs=(X-s.mu)/s.sd; n,d=Xs.shape; s.w=np.zeros(d); s.b=0.
        for _ in range(s.ep):
            p=1/(1+np.exp(-(Xs@s.w+s.b)))
            s.w-=s.lr*(Xs.T@(p-y)/n+s.l2*s.w); s.b-=s.lr*(p-y).mean()
        return s
    def predict(s,X): return (1/(1+np.exp(-(((X-s.mu)/s.sd)@s.w+s.b)))>=.5).astype(int)

N = min(len(real_paths), len(fake_paths), len(matched_paths), 60)
Xr = np.stack([radial_features(load_gray(p)) for p in tqdm(real_paths[:N])])
Xf = np.stack([radial_features(load_gray(p)) for p in tqdm(fake_paths[:N])])
Xm = np.stack([radial_features(load_gray(p)) for p in tqdm(matched_paths[:N])])

half = N//2
Xtr = np.vstack([Xr[:half], Xf[:half]])
ytr = np.r_[np.zeros(half), np.ones(half)]
det = LogReg().fit(Xtr, ytr)

clean_acc  = (det.predict(np.vstack([Xr[half:],Xf[half:]])) == np.r_[np.zeros(half),np.ones(half)]).mean()
evasion    = (det.predict(Xm[half:]) == 0).mean()
print(f'Radial detector  clean acc={clean_acc:.2f}  evasion rate={evasion:.2f}')

## 7. Build CSVs for SPAI inference

SPAI takes CSV files (`image,class,split`). We need one for raw fakes and one for matched fakes.

In [ ]:
import pandas as pd

def make_csv(paths, label, out_path):
    rows = [{'image': p, 'class': label, 'split': 'test'} for p in paths]
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(f'Wrote {len(rows)} rows -> {out_path}')

make_csv(real_paths[:50],    0, '/content/spai_real.csv')
make_csv(fake_paths[:50],    1, '/content/spai_fake.csv')
make_csv(matched_paths[:50], 1, '/content/spai_matched.csv')

## 8. SPAI inference — does the attack fool the SOTA detector?

We run `python -m spai infer` three times: on reals, raw fakes, and matched fakes.
The `--csv-root-dir /` is used because our CSVs contain absolute paths.

In [ ]:
import os
os.makedirs('/content/spai_out/real',    exist_ok=True)
os.makedirs('/content/spai_out/fake',    exist_ok=True)
os.makedirs('/content/spai_out/matched', exist_ok=True)

CFG    = '/content/spai/configs/spai.yaml'
WEIGHTS= '/content/spai/weights/spai.pth'

!python -m spai infer \
    --cfg {CFG} \
    --model {WEIGHTS} \
    --input /content/spai_real.csv \
    --csv-root-dir / \
    --output /content/spai_out/real \
    --tag real

!python -m spai infer \
    --cfg {CFG} \
    --model {WEIGHTS} \
    --input /content/spai_fake.csv \
    --csv-root-dir / \
    --output /content/spai_out/fake \
    --tag fake

!python -m spai infer \
    --cfg {CFG} \
    --model {WEIGHTS} \
    --input /content/spai_matched.csv \
    --csv-root-dir / \
    --output /content/spai_out/matched \
    --tag matched

print('SPAI inference done')

## 9. Results — the evasion table

In [ ]:
import pandas as pd, glob

def load_spai_results(tag):
    csvs = glob.glob(f'/content/spai_out/{tag}/**/*.csv', recursive=True)
    if not csvs:
        raise FileNotFoundError(f'No SPAI output CSV found for tag {tag}')
    return pd.read_csv(csvs[0])

res_real    = load_spai_results('real')
res_fake    = load_spai_results('fake')
res_matched = load_spai_results('matched')

# SPAI outputs a score (higher = more likely fake)
# threshold at 0.5 for label prediction
score_col = [c for c in res_fake.columns if 'score' in c.lower() or 'pred' in c.lower()][0]
print('Score column:', score_col)

fake_det_rate    = (res_fake[score_col]    >= 0.5).mean()
matched_det_rate = (res_matched[score_col] >= 0.5).mean()
evasion_spai     = 1.0 - matched_det_rate

print()
print('='*55)
print('freqgen — SPAI Evasion Table')
print('='*55)
print(f'Dataset:          Synthbuster SD1.4 vs RAISE-1k reals')
print(f'High-band gap     real/fake:    {real_high_mean/fake_high_mean:.1f}x')
print(f'High-band gap     real/matched: {real_high_mean/matched_high_mean:.2f}x')
print()
print(f'Hand-crafted radial detector:')
print(f'  clean acc={clean_acc:.2f}   evasion={evasion:.2f}')
print()
print(f'SPAI (CVPR 2025) detector:')
print(f'  raw fake detection rate:     {fake_det_rate:.2f}')
print(f'  matched fake detection rate: {matched_det_rate:.2f}')
print(f'  evasion rate (matched):      {evasion_spai:.2f}')
print('='*55)
print()
if evasion_spai > 0.5:
    print('RESULT: Attack EVADES SPAI -> gap found in CVPR 2025 SOTA')
else:
    print('RESULT: SPAI survives attack -> learned detectors are robust')

## 10. Save results figure + mount Drive

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Spectral Matching Attack: Real vs Fake vs Matched', fontsize=13, fontweight='bold')

# Radial profiles
prof_real = np.mean(real_profiles[:30], axis=0)
prof_fake = np.mean([radial_profile(load_gray(p)) for p in fake_paths[:30]], axis=0)
prof_matched = np.mean([radial_profile(load_gray(p)) for p in matched_paths[:30]], axis=0)

ax = axes[0]
ax.semilogy(prof_real,    color='blue',  lw=2, label='Real (RAISE-1k)')
ax.semilogy(prof_fake,    color='red',   lw=2, ls='--', label='Fake (SD1.4)')
ax.semilogy(prof_matched, color='green', lw=2, ls=':',  label='Matched')
ax.axvspan(0,  20,  alpha=0.07, color='green')
ax.axvspan(20, 60,  alpha=0.07, color='yellow')
ax.axvspan(60, 128, alpha=0.07, color='orange')
ax.set_title('Radial magnitude profile'); ax.set_xlabel('radius'); ax.legend()

# Band bar chart
bands = ['Low\n0-20', 'Mid\n20-60', 'High\n60+']
x = np.arange(3); w = 0.25
axes[1].bar(x-w, [spectral_report(load_gray(real_paths[0]))[k] for k in ('low','mid','high')], w, label='Real',    color='blue')
axes[1].bar(x,   [spectral_report(load_gray(fake_paths[0]))[k] for k in ('low','mid','high')], w, label='Fake',    color='red')
axes[1].bar(x+w, [spectral_report(load_gray(matched_paths[0]))[k] for k in ('low','mid','high')], w, label='Matched', color='green')
axes[1].set_yscale('log'); axes[1].set_xticks(x); axes[1].set_xticklabels(bands)
axes[1].set_title('Band energy'); axes[1].legend()

# SPAI score distribution
axes[2].hist(res_fake[score_col],    bins=20, alpha=0.6, color='red',   label='Fake (raw)')
axes[2].hist(res_matched[score_col], bins=20, alpha=0.6, color='green', label='Fake (matched)')
axes[2].hist(res_real[score_col],    bins=20, alpha=0.6, color='blue',  label='Real')
axes[2].axvline(0.5, color='black', ls='--', label='threshold')
axes[2].set_title('SPAI score distribution'); axes[2].set_xlabel('score (>0.5 = fake)'); axes[2].legend()

plt.tight_layout()
plt.savefig('/content/freqgen_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved /content/freqgen_result.png')

In [ ]:
# Optional: save to Google Drive
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy('/content/freqgen_result.png', '/content/drive/MyDrive/freqgen_result.png')
print('Saved to Google Drive')